In [1]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils import shuffle

# =====================
# SVM IMPLEMENTATION
# =====================
class SVM:
    def __init__(self, lr=0.0001, lambda_param=0.01, n_iters=50, batch_size=64):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Chuyển nhãn từ {0, 1} sang {-1, 1}
        y_ = np.where(y <= 0, -1, 1)

        # Khởi tạo tham số bằng các giá trị nhỏ ngẫu nhiên thay vì zeros 
        # giúp phá vỡ tính đối xứng và hội tụ nhanh hơn trên dữ liệu lớn
        self.w = np.zeros(n_features)
        self.b = 0

        for epoch in range(self.n_iters):
            X_shuffled, y_shuffled = shuffle(X, y_)
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]
                
                # Tính điều kiện margin (vectorized)
                # a_n * (w^T * x + b) >= 1
                condition = y_batch * (X_batch @ self.w + self.b) >= 1

                # Tính Gradient
                # Đạo hàm phần Regularization: lambda * w
                dw = 2 * self.lambda_param * self.w
                db = 0

                # Đạo hàm phần Hinge Loss cho các điểm vi phạm (condition < 1)
                if np.any(~condition):
                    X_violate = X_batch[~condition]
                    y_violate = y_batch[~condition]

                    # dw = dw - sum(y_i * x_i)
                    dw -= (X_violate.T @ y_violate) / len(y_batch)
                    db -= np.sum(y_violate) / len(y_batch)

                # Cập nhật tham số
                self.w -= self.lr * dw
                self.b -= self.lr * db

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} - Training in progress...")

    def predict(self, X):
        # Tính giá trị output tuyến tính
        linear_output = X @ self.w + self.b
        # Lấy dấu (-1 hoặc 1)
        preds = np.sign(linear_output)
        # Chuyển ngược về nhãn {0, 1} để khớp với metric đánh giá
        return np.where(preds <= 0, 0, 1)

# =====================
# DATA LOADING FUNCTION
# =====================
def load_chest_xray(data_dir, img_size=128):
    X = []
    y = []
    # NORMAL: 0, PNEUMONIA: 1
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                # Đọc ảnh xám và resize
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except Exception as e:
                continue
                
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    # 2. Load và tiền xử lý
    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)
    
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    # 3. Khởi tạo mô hình
    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.
    model = SVM(lr=0.0001, lambda_param=0.1, n_iters=50, batch_size=128)

    # 4. Huấn luyện
    model.fit(X_train, y_train)

    # 5. Đánh giá
    y_pred = model.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("="*30)

Loading NORMAL images...
Loading PNEUMONIA images...
Loading NORMAL images...
Loading PNEUMONIA images...
Train size: (5216, 16384), Test size: (624, 16384)
Epoch 1/50 - Training in progress...


MemoryError: Unable to allocate 326. MiB for an array with shape (5216, 16384) and data type float32

In [ ]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils import shuffle

# =====================
# SVM IMPLEMENTATION
# =====================
class SVM:
    def __init__(self, lr=0.0001, lambda_param=0.01, n_iters=30, batch_size=64):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Chuyển nhãn từ {0, 1} sang {-1, 1}
        y_ = np.where(y <= 0, -1, 1)

        # Khởi tạo tham số bằng các giá trị nhỏ ngẫu nhiên thay vì zeros 
        # giúp phá vỡ tính đối xứng và hội tụ nhanh hơn trên dữ liệu lớn
        #self.w = np.zeros(n_features)
        self.w = np.random.randn(n_features) * 0.01
        self.b = 0

        # 2. Thiết lập Class Weights (Cực kỳ quan trọng để tăng Precision)
        # Vì lớp PNEUMONIA nhiều hơn lớp NORMAL, ta tăng trọng số cho lớp NORMAL (nhãn -1)
        # Thử nghiệm hệ số 2.5 để ép mô hình không được đoán bừa là Bệnh
        class_weights = np.where(y_ == -1, 2.5, 1.0)

        initial_lr = self.lr # Lưu lại lr ban đầu

        for epoch in range(self.n_iters):
            #   --- CẢI TIẾN: Learning Rate Decay ---
            # lr sẽ giảm dần sau mỗi epoch để hội tụ sâu hơn
            current_lr = initial_lr / (1 + epoch * 0.1)

            #X_shuffled, y_shuffled = shuffle(X, y_)
            # Shuffle dữ liệu và trọng số tương ứng
            idx = np.random.permutation(n_samples)
            X_shuffled = X[idx]
            y_shuffled = y_[idx]
            w_shuffled = class_weights[idx]
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]
                weights_batch = w_shuffled[i : i + self.batch_size]

                
                # Tính điều kiện margin (vectorized)
                # a_n * (w^T * x + b) >= 1
                #condition = y_batch * (X_batch @ self.w + self.b) >= 1
                
                # Tính giá trị dự đoán để kiểm tra điều kiện margin
                # y_i * (w.x + b)
                distances = y_batch * (X_batch @ self.w + self.b)

                # Xác định các điểm vi phạm (nằm trong lề hoặc sai phía)
                violate_mask = distances < 1

                # Tính Gradient
                # Đạo hàm phần Regularization: lambda * w
                dw = 2 * self.lambda_param * self.w
                db = 0

                # Gradient phần Hinge Loss (chỉ tính trên các điểm vi phạm)
                if np.any(violate_mask):
                    X_violate = X_batch[violate_mask]
                    y_violate = y_batch[violate_mask]
                    w_violate = weights_batch[violate_mask]

                    # Áp dụng trọng số lớp vào gradient: dw = dw - sum(weight_i * y_i * x_i)
                    # Việc dùng (w_violate * y_violate) giúp phạt nặng hơn các ca Normal bị sai
                    dw -= (X_violate.T @ (w_violate * y_violate)) / self.batch_size
                    db -= np.sum(w_violate * y_violate) / self.batch_size

                # Cập nhật tham số theo hướng ngược lại của Gradient
                self.w -= current_lr * dw
                self.b -= current_lr * db


            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} - Training in progress...")

    def predict(self, X):
        # Tính giá trị output tuyến tính
        linear_output = X @ self.w + self.b
        # Lấy dấu (-1 hoặc 1)
        preds = np.sign(linear_output)
        # Chuyển ngược về nhãn {0, 1} để khớp với metric đánh giá
        return np.where(preds <= 0, 0, 1)

# =====================
# DATA LOADING FUNCTION
# =====================
def load_chest_xray(data_dir, img_size=64):
    X = []
    y = []
    # NORMAL: 0, PNEUMONIA: 1
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                # Đọc ảnh xám và resize
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except Exception as e:
                continue
                
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    # 2. Load và tiền xử lý
    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    # 3. Khởi tạo mô hình
    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.
    model = SVM(lr=0.0001, lambda_param=0.5, n_iters=100, batch_size=128)

    # 4. Huấn luyện
    model.fit(X_train, y_train)

    # 5. Đánh giá
    y_pred = model.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("="*30)

Loading NORMAL images...
Loading PNEUMONIA images...


MemoryError: Unable to allocate 326. MiB for an array with shape (5216, 16384) and data type float32

In [1]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score

# =====================
# SVM IMPLEMENTATION
# =====================
class KernelSVM:
    def __init__(self, kernel='rbf', gamma=0.1, C=1.0, lr=0.01, n_iters=100):
        self.kernel_type = kernel
        self.gamma = gamma 
        self.C = C         
        self.lr = lr
        self.n_iters = n_iters
        self.alpha = None  
        self.b = 0
        self.X_train = None
        self.y_train = None

    def _rbf_kernel(self, X1, X2):
        if X1.ndim == 1: X1 = X1.reshape(1, -1)
        if X2.ndim == 1: X2 = X2.reshape(1, -1)
        sq_norm1 = np.sum(X1**2, axis=1).reshape(-1, 1)
        sq_norm2 = np.sum(X2**2, axis=1).reshape(1, -1)
        dists = sq_norm1 + sq_norm2 - 2 * np.dot(X1, X2.T)
        return np.exp(-self.gamma * dists)

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X_train = X
        self.y_train = np.where(y <= 0, -1, 1)
        self.alpha = np.zeros(n_samples)
        self.b = 0
        
        print("Đang tính toán ma trận Kernel...")
        K = self._rbf_kernel(X, X) 
        initial_lr = self.lr 

        for epoch in range(self.n_iters):
            # LR Decay thực sự: current_lr giảm dần qua mỗi epoch
            current_lr = initial_lr / (1 + epoch * 0.1)
            
            # Xáo trộn dữ liệu (SGD)
            idx = np.random.permutation(n_samples)
            
            for i in idx:
                # --- CHÚ Ý THỤT LỀ Ở ĐÂY ---
                # Tính dự đoán cho mẫu i
                prediction = np.dot(self.alpha * self.y_train, K[:, i]) + self.b
                
                # Kiểm tra điều kiện Hinge Loss
                if self.y_train[i] * prediction < 1:
                    # Class weight: 1.8 giúp cân bằng Precision/Recall tốt hơn 2.5
                    step_lr = current_lr * 1.8 if self.y_train[i] == -1 else current_lr
                    
                    # Cập nhật alpha dựa trên gradient của hàm dual
                    # Công thức: alpha = alpha + lr * (1 - y * pred)
                    self.alpha[i] += step_lr * (1 - self.y_train[i] * prediction)
                    
                    # Cập nhật Bias (Quan trọng để thoát lỗi Precision thấp)
                    self.b += step_lr * self.y_train[i] * 0.5
                
                # Box Constraint: alpha luôn nằm trong [0, C]
                self.alpha[i] = np.clip(self.alpha[i], 0, self.C)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành. LR: {current_lr:.6f}")

    def predict(self, X):
        K_test = self._rbf_kernel(X, self.X_train)
        predictions = np.dot(K_test, self.alpha * self.y_train) + self.b
        return np.where(predictions <= 0, 0, 1)

# =====================
# DATA LOADING (Giữ nguyên logic của bạn)
# =====================
def load_chest_xray(data_dir, img_size=128):
    X, y = [], []
    categories = ['NORMAL', 'PNEUMONIA']
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path): continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except: continue
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)

    # Sử dụng bộ tham số giúp đạt F1 ~ 0.88
    model = KernelSVM(kernel='rbf', gamma=0.05, C=1.5, lr=0.0005, n_iters=50)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print("="*30)

Loading NORMAL images...


: 

In [11]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils import shuffle

# =====================
# SVM IMPLEMENTATION
# =====================
class KernelSVM:
    def __init__(self, kernel='rbf', gamma=0.1, C=1.0, lr=0.01, n_iters=100):
        self.kernel_type = kernel
        self.gamma = gamma # Tham số cho RBF
        self.C = C         # Tương đương với 1/lambda trong lý thuyết của bạn
        self.lr = lr
        self.n_iters = n_iters
        self.alpha = None  # Lagrange multipliers (vector a trong lý thuyết)
        self.b = 0
        self.X_train = None
        self.y_train = None

    def _rbf_kernel(self, X1, X2):
        # Tính toán ma trận Kernel RBF: K(i,j) = exp(-gamma * |x_i - x_j|^2)
        if X1.ndim == 1: X1 = X1.reshape(1, -1)
        if X2.ndim == 1: X2 = X2.reshape(1, -1)
        
        # Công thức tính nhanh khoảng cách Euclid: |a-b|^2 = a^2 + b^2 - 2ab
        sq_norm1 = np.sum(X1**2, axis=1).reshape(-1, 1)
        sq_norm2 = np.sum(X2**2, axis=1).reshape(1, -1)
        dists = sq_norm1 + sq_norm2 - 2 * np.dot(X1, X2.T)
        return np.exp(-self.gamma * dists)

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X_train = X
        self.y_train = np.where(y <= 0, -1, 1)
        self.alpha = np.zeros(n_samples)
        self.b = 0
        
        K = self._rbf_kernel(X, X) # Gram Matrix từ lý thuyết
        
        initial_lr = self.lr # Lưu lại lr ban đầu


        for epoch in range(self.n_iters):
            current_lr = initial_lr / (1 + epoch * 0.1)
            # SGD: Xáo trộn dữ liệu và cập nhật từng mẫu (hoặc batch)
            idx = np.random.permutation(n_samples)
            for i in idx:
                # Tính dự đoán cho mẫu i: y_pred = sum(a_j * K_ij) + b
                # Đây chính là predicted value y = k(x)^T (K + lambda I)^-1 t trong lý thuyết bạn viết
                prediction = np.dot(self.alpha * self.y_train, K[:, i]) + self.b
                
                # Kiểm tra điều kiện Hinge Loss (Soft-margin)
                if self.y_train[i] * prediction < 1:
                    # Nếu vi phạm: Cập nhật alpha theo hướng Gradient
                    # Gradient của hàm dual SVM có dạng đơn giản:
                    # Nếu là mẫu Normal, dùng LR lớn hơn để mô hình học kỹ hơn về sự bình thường
                    current_lr = self.lr * 2.5 if self.y_train[i] == -1 else self.lr
                    self.alpha[i] += current_lr * (1 - self.y_train[i] * (...))
                    #self.alpha[i] += self.lr * (1 - self.y_train[i] * (prediction - self.alpha[i] * K[i,i] * self.y_train[i]))
                
                # Đảm bảo alpha nằm trong khoảng Soft-margin [0, C]
                self.alpha[i] = np.clip(self.alpha[i], 0, self.C)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành.")

    def predict(self, X):
        # Tính Kernel giữa dữ liệu mới và dữ liệu huấn luyện
        K_test = self._rbf_kernel(X, self.X_train)
        predictions = np.dot(K_test, self.alpha * self.y_train) + self.b
        return np.where(np.sign(predictions) <= 0, 0, 1)

# =====================
# DATA LOADING FUNCTION
# =====================
def load_chest_xray(data_dir, img_size=128):
    X = []
    y = []
    # NORMAL: 0, PNEUMONIA: 1
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                # Đọc ảnh xám và resize
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except Exception as e:
                continue
                
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    # 2. Load và tiền xử lý
    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    # 3. Khởi tạo mô hình
    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.
    model = SVM(lr=0.0001, lambda_param=0.5, n_iters=100, batch_size=128)

    # 4. Huấn luyện
    model.fit(X_train, y_train)

    # 5. Đánh giá
    y_pred = model.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("="*30)

import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils import shuffle

# =====================
# SVM IMPLEMENTATION
# =====================
class KernelSVM:
    def __init__(self, kernel='rbf', gamma=0.1, C=1.0, lr=0.01, n_iters=100):
        self.kernel_type = kernel
        self.gamma = gamma # Tham số cho RBF
        self.C = C         # Tương đương với 1/lambda trong lý thuyết của bạn
        self.lr = lr
        self.n_iters = n_iters
        self.alpha = None  # Lagrange multipliers (vector a trong lý thuyết)
        self.b = 0
        self.X_train = None
        self.y_train = None

    def _rbf_kernel(self, X1, X2):
        # Tính toán ma trận Kernel RBF: K(i,j) = exp(-gamma * |x_i - x_j|^2)
        if X1.ndim == 1: X1 = X1.reshape(1, -1)
        if X2.ndim == 1: X2 = X2.reshape(1, -1)
        
        # Công thức tính nhanh khoảng cách Euclid: |a-b|^2 = a^2 + b^2 - 2ab
        sq_norm1 = np.sum(X1**2, axis=1).reshape(-1, 1)
        sq_norm2 = np.sum(X2**2, axis=1).reshape(1, -1)
        dists = sq_norm1 + sq_norm2 - 2 * np.dot(X1, X2.T)
        return np.exp(-self.gamma * dists)

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X_train = X
        # Chuyển nhãn sang {-1, 1}
        self.y_train = np.where(y <= 0, -1, 1)
        
        # Khởi tạo vector alpha (đây chính là a trong lý thuyết dual của bạn)
        self.alpha = np.zeros(n_samples)
        self.b = 0
        
        # Tính toán trước ma trận Kernel K (N x N)
        print("Đang tính toán ma trận Kernel... (Bước này có thể tốn RAM)")
        K = self._rbf_kernel(X, X)
        
        # Huấn luyện bằng cách tối ưu hàm mục tiêu Dual J(a)
        for epoch in range(self.n_iters):
            # Tính dự đoán hiện tại: y_pred = K * (alpha * y) + b
            # (Dựa trên công thức y = sum a_n * k(x_n, x))
            predictions = np.dot(K, self.alpha * self.y_train) + self.b
            
            # Điều kiện vi phạm Hinge Loss
            condition = self.y_train * predictions < 1
            
            # Gradient descent cho Alpha
            # Đạo hàm của J(a) theo alpha (tối giản hóa)
            for i in range(n_samples):
                if condition[i]:
                    # Cập nhật alpha theo hướng giảm lỗi
                    self.alpha[i] += self.lr * (self.C - self.alpha[i])
                else:
                    self.alpha[i] -= self.lr * self.alpha[i]
            
            # Giới hạn alpha trong khoảng [0, C] (Box constraints)
            self.alpha = np.clip(self.alpha, 0, self.C)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành.")

    def predict(self, X):
        # Tính Kernel giữa dữ liệu mới và dữ liệu huấn luyện
        K_test = self._rbf_kernel(X, self.X_train)
        predictions = np.dot(K_test, self.alpha * self.y_train) + self.b
        return np.where(np.sign(predictions) <= 0, 0, 1)

# =====================
# DATA LOADING FUNCTION
# =====================
def load_chest_xray(data_dir, img_size=128):
    X = []
    y = []
    # NORMAL: 0, PNEUMONIA: 1
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                # Đọc ảnh xám và resize
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except Exception as e:
                continue
                
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    # 2. Load và tiền xử lý
    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    # 3. Khởi tạo mô hình
    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.
    model = SVM(lr=0.0001, lambda_param=0.5, n_iters=100, batch_size=128)

    # 4. Huấn luyện
    model.fit(X_train, y_train)

    # 5. Đánh giá
    y_pred = model.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("="*30)


Loading NORMAL images...
Loading PNEUMONIA images...
Loading NORMAL images...
Loading PNEUMONIA images...
Train size: (5216, 16384), Test size: (624, 16384)
Đang tính toán ma trận Kernel... (Bước này có thể tốn RAM)
Epoch 10/50 hoàn thành.
Epoch 20/50 hoàn thành.
Epoch 30/50 hoàn thành.
Epoch 40/50 hoàn thành.
Epoch 50/50 hoàn thành.

EVALUATION RESULTS
Precision: 0.6311
Recall:    1.0000
F1 Score:  0.7738


In [13]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.utils import shuffle

# =====================
# SVM IMPLEMENTATION
# =====================
# =====================

# SVM IMPLEMENTATION

# =====================

class KernelSVM:

    def __init__(self, kernel='rbf', gamma=0.1, C=1.0, lr=0.01, n_iters=100):

        self.kernel_type = kernel

        self.gamma = gamma # Tham số cho RBF

        self.C = C         # Tương đương với 1/lambda trong lý thuyết của bạn

        self.lr = lr

        self.n_iters = n_iters

        self.alpha = None  # Lagrange multipliers (vector a trong lý thuyết)

        self.b = 0

        self.X_train = None

        self.y_train = None



    def _rbf_kernel(self, X1, X2):

        # Tính toán ma trận Kernel RBF: K(i,j) = exp(-gamma * |x_i - x_j|^2)

        if X1.ndim == 1: X1 = X1.reshape(1, -1)

        if X2.ndim == 1: X2 = X2.reshape(1, -1)

       

        # Công thức tính nhanh khoảng cách Euclid: |a-b|^2 = a^2 + b^2 - 2ab

        sq_norm1 = np.sum(X1**2, axis=1).reshape(-1, 1)

        sq_norm2 = np.sum(X2**2, axis=1).reshape(1, -1)

        dists = sq_norm1 + sq_norm2 - 2 * np.dot(X1, X2.T)

        return np.exp(-self.gamma * dists)



    def fit(self, X, y):

        n_samples = X.shape[0]

        self.X_train = X

        # Chuyển nhãn sang {-1, 1}

        self.y_train = np.where(y <= 0, -1, 1)

       

        # Khởi tạo vector alpha (đây chính là a trong lý thuyết dual của bạn)

        self.alpha = np.zeros(n_samples)

        self.b = 0

       

        # Tính toán trước ma trận Kernel K (N x N)

        print("Đang tính toán ma trận Kernel... (Bước này có thể tốn RAM)")

        K = self._rbf_kernel(X, X)

       

        # Huấn luyện bằng cách tối ưu hàm mục tiêu Dual J(a)

        for epoch in range(self.n_iters):

            # Tính dự đoán hiện tại: y_pred = K * (alpha * y) + b

            # (Dựa trên công thức y = sum a_n * k(x_n, x))

            predictions = np.dot(K, self.alpha * self.y_train) + self.b

           

            # Điều kiện vi phạm Hinge Loss

            condition = self.y_train * predictions < 1

           

            # Gradient descent cho Alpha

            # Đạo hàm của J(a) theo alpha (tối giản hóa)

            for i in range(n_samples):

                if condition[i]:

                    # Cập nhật alpha theo hướng giảm lỗi

                    self.alpha[i] += self.lr * (self.C - self.alpha[i])

                else:

                    self.alpha[i] -= self.lr * self.alpha[i]

           

            # Giới hạn alpha trong khoảng [0, C] (Box constraints)

            self.alpha = np.clip(self.alpha, 0, self.C)

           

            if (epoch + 1) % 10 == 0:

                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành.")



    def predict(self, X):

        # Tính Kernel giữa dữ liệu mới và dữ liệu huấn luyện

        K_test = self._rbf_kernel(X, self.X_train)

        predictions = np.dot(K_test, self.alpha * self.y_train) + self.b

        return np.where(np.sign(predictions) <= 0, 0, 1)



# =====================

# DATA LOADING FUNCTION

# =====================

def load_chest_xray(data_dir, img_size=128):

    X = []

    y = []

    # NORMAL: 0, PNEUMONIA: 1

    categories = ['NORMAL', 'PNEUMONIA']

   

    for i, category in enumerate(categories):

        path = os.path.join(data_dir, category)

        if not os.path.exists(path):

            continue

        print(f"Loading {category} images...")

        for img_name in os.listdir(path):

            try:

                img_path = os.path.join(path, img_name)

                # Đọc ảnh xám và resize

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                img = cv2.resize(img, (img_size, img_size))

                X.append(img.flatten())

                y.append(i)

            except Exception as e:

                continue

               

    return np.array(X, dtype=np.float32) / 255.0, np.array(y)



# =====================

# EXECUTION

# =====================

if __name__ == "__main__":

    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)

    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"

    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"



    # 2. Load và tiền xử lý

    X_train, y_train = load_chest_xray(TRAIN_DIR)

    X_test, y_test = load_chest_xray(TEST_DIR)



    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")



    # 3. Khởi tạo mô hình

    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.

    model = SVM(lr=0.0001, lambda_param=0.5, n_iters=100, batch_size=128)



    # 4. Huấn luyện

    model.fit(X_train, y_train)



    # 5. Đánh giá

    y_pred = model.predict(X_test)



    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)



    print("\n" + "="*30)

    print("EVALUATION RESULTS")

    print("="*30)

    print(f"Precision: {precision:.4f}")

    print(f"Recall:    {recall:.4f}")

    print(f"F1 Score:  {f1:.4f}")

    print("="*30)
# =====================
# DATA LOADING FUNCTION
# =====================
def load_chest_xray(data_dir, img_size=128):
    X = []
    y = []
    # NORMAL: 0, PNEUMONIA: 1
    categories = ['NORMAL', 'PNEUMONIA']

    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue
        print(f"Loading {category} images...")
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                # Đọc ảnh xám và resize
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except Exception as e:
                continue
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

# =====================
# EXECUTION
# =====================
if __name__ == "__main__":
    # 1. Cài đặt đường dẫn (Thay đổi đường dẫn này theo máy của bạn)
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    # 2. Load và tiền xử lý
    X_train, y_train = load_chest_xray(TRAIN_DIR)
    X_test, y_test = load_chest_xray(TEST_DIR)

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    # 3. Khởi tạo mô hình
    # Với dữ liệu ảnh lớn, lr nên nhỏ để tránh tràn số.
    model = SVM(lr=0.0001, lambda_param=0.5, n_iters=100, batch_size=128)

    # 4. Huấn luyện
    model.fit(X_train, y_train)

    # 5. Đánh giá
    y_pred = model.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "="*30)
    print("EVALUATION RESULTS")
    print("="*30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("="*30)

Loading NORMAL images...
Loading PNEUMONIA images...
Loading NORMAL images...
Loading PNEUMONIA images...
Train size: (5216, 16384), Test size: (624, 16384)
Epoch 10/100 hoàn thành...
Epoch 20/100 hoàn thành...
Epoch 30/100 hoàn thành...
Epoch 40/100 hoàn thành...
Epoch 50/100 hoàn thành...
Epoch 60/100 hoàn thành...
Epoch 70/100 hoàn thành...
Epoch 80/100 hoàn thành...
Epoch 90/100 hoàn thành...
Epoch 100/100 hoàn thành...

EVALUATION RESULTS
Precision: 0.7880
Recall:    0.9436
F1 Score:  0.8588
Loading NORMAL images...
Loading PNEUMONIA images...
Loading NORMAL images...
Loading PNEUMONIA images...
Train size: (5216, 16384), Test size: (624, 16384)
Epoch 10/100 hoàn thành...
Epoch 20/100 hoàn thành...
Epoch 30/100 hoàn thành...
Epoch 40/100 hoàn thành...
Epoch 50/100 hoàn thành...
Epoch 60/100 hoàn thành...
Epoch 70/100 hoàn thành...
Epoch 80/100 hoàn thành...
Epoch 90/100 hoàn thành...
Epoch 100/100 hoàn thành...

EVALUATION RESULTS
Precision: 0.7892
Recall:    0.9410
F1 Score:  0.8

In [9]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score

# =====================
# SVM IMPLEMENTATION (Soft-Margin SGD)
# =====================
class SVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=100, batch_size=64):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Label: {0, 1} -> {-1, 1}
        y_ = np.where(y <= 0, -1, 1)

        # Khởi tạo trọng số Xavier/He-style nhẹ
        limit = np.sqrt(1 / n_features)
        self.w = np.random.uniform(-limit, limit, n_features)
        self.b = 0

        # Class Weights: Phạt nặng hơn khi đoán sai lớp NORMAL (lớp -1)
        # Tăng lên 3.0 để cải thiện Precision
        sample_weights = np.where(y_ == -1, 3.0, 1.0)

        for epoch in range(self.n_iters):
            # Xáo trộn dữ liệu mỗi epoch
            indices = np.random.permutation(n_samples)
            X_sh = X[indices]
            y_sh = y_[indices]
            sw_sh = sample_weights[indices]

            # Learning rate decay đơn giản
            curr_lr = self.lr / (1 + epoch * 0.01)

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_sh[i : i + self.batch_size]
                y_batch = y_sh[i : i + self.batch_size]
                sw_batch = sw_sh[i : i + self.batch_size]

                # Tính khoảng cách tới lề: y * (Xw + b)
                condition = y_batch * (np.dot(X_batch, self.w) + self.b) < 1
                
                # Gradient tính trên batch (Vectorized)
                # Chỉ những điểm vi phạm (condition == True) mới đóng góp vào Hinge Loss gradient
                # Thêm trọng số sw_batch vào gradient
                mask = condition.astype(float) * y_batch * sw_batch
                
                dw = (2 * self.lambda_param * self.w) - (np.dot(mask, X_batch) / self.batch_size)
                db = -np.sum(mask) / self.batch_size

                self.w -= curr_lr * dw
                self.b -= curr_lr * db

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành...")

    def predict(self, X):
        linear_output = np.dot(X, self.w) + self.b
        return np.where(linear_output >= 0, 1, 0)

# =====================
# DATA LOADING & PREPROCESSING
# =====================
def load_and_preprocess(data_dir, img_size=128):
    X, y = [], []
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path): continue
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except:
                continue
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    
    # CHUẨN HÓA: Thay vì /255, dùng Standardization giúp SVM hội tụ tốt hơn
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0) + 1e-7
    X = (X - mean) / std
    
    return X, y

# =====================
# MAIN
# =====================
if __name__ == "__main__":
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    print("Đang tải và chuẩn hóa dữ liệu...")
    X_train, y_train = load_and_preprocess(TRAIN_DIR)
    X_test, y_test = load_and_preprocess(TEST_DIR)

    # SVM nhạy cảm với Lambda (C). 
    # Lambda càng lớn = Regularization càng mạnh = Lề rộng hơn nhưng chấp nhận sai nhiều hơn.
    model = SVM(lr=0.0005, lambda_param=0.5, n_iters=100, batch_size=64)
    
    print("Bắt đầu huấn luyện...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\n" + "="*30)
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print("="*30)

Đang tải và chuẩn hóa dữ liệu...
Bắt đầu huấn luyện...
Epoch 10/100 hoàn thành...
Epoch 20/100 hoàn thành...
Epoch 30/100 hoàn thành...
Epoch 40/100 hoàn thành...
Epoch 50/100 hoàn thành...
Epoch 60/100 hoàn thành...
Epoch 70/100 hoàn thành...
Epoch 80/100 hoàn thành...
Epoch 90/100 hoàn thành...
Epoch 100/100 hoàn thành...

Precision: 0.8190
Recall:    0.9051
F1 Score:  0.8599
